# Aspect-Based Sentiment Analysis for Restaurant Ranking

## Load Data

In [162]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import glob

# Project directory
project_dir = os.path.dirname(os.getcwd())

# Raw Google data folder
google_dir = os.path.join(
    project_dir,
    "data",
    "raw_data",
    "GoogleReview"
)

# Find all Google CSV files
google_files = glob.glob(
    os.path.join(google_dir, "GoogleReview_*.csv")
)

print("Google files found:")
for file in google_files:
    print(os.path.basename(file))

Google files found:
GoogleReview_Ipoh.csv
GoogleReview_JB.csv
GoogleReview_KL.csv
GoogleReview_Kuching.csv
GoogleReview_Langkawi.csv
GoogleReview_Melaka.csv
GoogleReview_Miri.csv
GoogleReview_Penang.csv
GoogleReview_Petaling Jaya.csv
GoogleReview_Shah Alam.csv


In [163]:
def extract_google_location(filepath):
    filename = os.path.basename(filepath)

    # GoogleReview_Miri.csv → Miri
    location = filename.replace("GoogleReview_", "")
    location = os.path.splitext(location)[0]

    return location

In [164]:
google_dfs = []

for file in google_files:
    temp_df = pd.read_csv(file)

    # Extract location
    location = extract_google_location(file)

    # Add platform and location
    temp_df["Platform"] = "Google"
    temp_df["Location"] = location

    google_dfs.append(temp_df)

# Combine
df_google = pd.concat(
    google_dfs,
    ignore_index=True
)

print("Combined Google dataset shape:", df_google.shape)

Combined Google dataset shape: (231112, 6)


In [165]:
# Clean Rating column
df_google["Rating"] = (
    df_google["Rating"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

In [166]:
google_output = os.path.join(
    project_dir,
    "data",
    "raw_data",
    "GoogleReview_data.csv"
)

df_google.to_csv(
    google_output,
    index=False
)

print("Saved:", google_output)

Saved: c:\Users\user\Downloads\social_computing_assignment\data\raw_data\GoogleReview_data.csv


In [167]:
tripadvisor_dir = os.path.join(
    project_dir,
    "data",
    "raw_data",
    "TripAdvisor"
)

tripadvisor_files = glob.glob(
    os.path.join(tripadvisor_dir, "reviews_all_*.csv")
)

print("TripAdvisor files found:")
for file in tripadvisor_files:
    print(os.path.basename(file))

TripAdvisor files found:
reviews_all_Ipoh.csv
reviews_all_JB.csv
reviews_all_KL.csv
reviews_all_Kuching.csv
reviews_all_Langkawi.csv
reviews_all_Melaka.csv
reviews_all_Miri.csv
reviews_all_Penang.csv
reviews_all_Petaling Jaya.csv
reviews_all_Shah Alam.csv


In [168]:
def extract_tripadvisor_location(filepath):
    filename = os.path.basename(filepath)

    # reviews_all_Ipoh.csv → Ipoh
    location = filename.replace("reviews_all_", "")
    location = os.path.splitext(location)[0]

    return location

In [169]:
tripadvisor_dfs = []

for file in tripadvisor_files:
    temp_df = pd.read_csv(file)

    # Extract location
    location = extract_tripadvisor_location(file)

    # Add platform and location
    temp_df["Platform"] = "Tripadvisor"
    temp_df["Location"] = location

    tripadvisor_dfs.append(temp_df)

# Combine
df_tripadvisor = pd.concat(
    tripadvisor_dfs,
    ignore_index=True
)

print(
    "Combined TripAdvisor dataset shape:",
    df_tripadvisor.shape
)

Combined TripAdvisor dataset shape: (140004, 8)


In [170]:
tripadvisor_output = os.path.join(
    project_dir,
    "data",
    "raw_data",
    "TripAdvisor_data.csv"
)

df_tripadvisor.to_csv(
    tripadvisor_output,
    index=False
)

print("Saved:", tripadvisor_output)

Saved: c:\Users\user\Downloads\social_computing_assignment\data\raw_data\TripAdvisor_data.csv


## Preprocessing

In [171]:
df_google.count()

Author        231112
Rating        231112
Review        223258
Restaurant    231112
Location      231112
Platform      231112
dtype: int64

In [172]:
df_tripadvisor.count()

Author        140004
Title         140004
Review        140004
Rating        140004
Dates         140004
Restaurant    140004
Platform      140004
Location      140004
dtype: int64

In [173]:
df_google["Platform"] = "Google"
df_tripadvisor["Platform"] = "Tripadvisor"

df = pd.concat(
    [df_google, df_tripadvisor],
    ignore_index=True
)

df.count()

Author        371116
Rating        371116
Review        363262
Restaurant    371116
Location      371116
Platform      371116
Title         140004
Dates         140004
dtype: int64

### Basic Cleaning

In [174]:
import emoji
import re
import contractions

def basic_cleaning(df, keep_emoji_text=True):

    # Make a copy to avoid modifying the original DataFrame
    df = df.copy()

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Drop rows where Rating is NaN
    df = df[df["Rating"].notna()].copy()

    # If Rating contains string placeholders like "", "nan", "none", "null", "n/a"
    invalid_ratings = ["", "nan", "none", "null", "n/a", "na"]
    df = df[
        ~df["Rating"].astype(str).str.strip().str.lower().isin(invalid_ratings)
    ].copy()

    # Remove actual NaN reviews
    df = df[df["Review"].notna()].copy()

    # Convert Review to string
    df["Review"] = df["Review"].astype(str)

    # Remove common textual missing values
    invalid_reviews = ["", "nan", "none", "null", "n/a", "na"]

    df = df[
        ~df["Review"].str.strip().str.lower().isin(invalid_reviews)
    ].copy()

    # Convert text to lowercase
    df["Review"] = df["Review"].str.lower()

    # Remove HTML tags
    df["Review"] = df["Review"].apply(
        lambda x: re.sub(r'<[^>]+>', '', x)
    )

    # Remove URLs
    df["Review"] = df["Review"].apply(
        lambda x: re.sub(r'http\S+|www\S+', '[URL]', x)
    )

    # Remove email addresses
    df["Review"] = df["Review"].apply(
        lambda x: re.sub(r'\S+@\S+', '[EMAIL]', x)
    )

    # Keep or remove emojis
    if keep_emoji_text:
        df["Review"] = df["Review"].apply(
            lambda x: emoji.demojize(
                x,
                delimiters=(' ', ' ')
            )
        )
    else:
        df["Review"] = df["Review"].apply(
            lambda x: emoji.replace_emoji(x, replace='')
        )

    # Expand contractions
    df["Review"] = df["Review"].apply(
        lambda x: contractions.fix(x)
    )

    # Remove mentions
    df["Review"] = df["Review"].apply(
        lambda x: re.sub(r'@\w+', '', x)
    )

    # Normalize whitespace
    df["Review"] = df["Review"].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip()
    )

    # Remove rows that became empty after cleaning
    df = df[
        df["Review"].notna() &
        df["Review"].str.strip().ne("")
    ].copy()

    # Reset index
    df.reset_index(drop=True, inplace=True)

    return df

In [175]:
# clean the dataframes
df_cleaned = basic_cleaning(df)

In [176]:
# Remove the "(translated by google)" text from the Review column
df_cleaned["Review"] = df_cleaned["Review"].apply(
    lambda x: re.sub(r'\s*\(translated by google\)', '', x)
)

In [177]:
# number of records
sampleNum = df_cleaned.index.size
print(f"Sample Number: {sampleNum}")

Sample Number: 361780


In [178]:
# subset wanted columns
def subset_columns(df):
    df = df[["Review", "Rating", "Restaurant","Platform"]]
    return df  

In [179]:
# remain only the wanted columns
df_cleaned = subset_columns(df_cleaned)

### Tokenization

In [180]:
from nltk.tokenize import sent_tokenize, word_tokenize

def tokenize_review(text):
    # Sentence tokenization
    sentences = sent_tokenize(text)
    
    # Word tokenization
    tokens = []
    for sentence in sentences:
        tokens.extend(word_tokenize(sentence))
    
    return sentences, tokens

In [181]:
df_cleaned[["Sentences", "Tokens"]] = df_cleaned["Review"].apply(
    lambda x: pd.Series(tokenize_review(x))
)

### Stop-word removal

In [182]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords

# Base English stopword list
stop_words = set(stopwords.words("english"))

# Words that carry sentiment, negation, or contrast
important_words = {
    "not",
    "no",
    "nor",
    "never",
    "neither",
    "none",
    "but",
    "however",
    "although",
    "yet",
    "though",
    "very",
    "too",
    "so",
    "really",
    "extremely",
    "quite",
    "highly",
    "somewhat",
    "slightly",
    "badly",
    "hardly",
    "absolutely",
    "completely",
    "could",
    "would",
    "should",
    "might",
    "must"
}

# Keep important words by removing them from stopword list
stop_words = stop_words - important_words

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token.lower() not in stop_words
    ]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [183]:
df_cleaned["Tokens_no_stopwords"] = df_cleaned["Tokens"].apply(remove_stopwords)

### Lemmatization

In [184]:
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [185]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [186]:
lemmatizer = WordNetLemmatizer()

In [187]:
def lemmatize_tokens(tokens):
    tagged_tokens = pos_tag(tokens)

    lemmatized = []

    for word, tag in tagged_tokens:
        wordnet_pos = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, pos=wordnet_pos)
        lemmatized.append(lemma)

    return lemmatized

In [188]:
df_cleaned["Lemmatized_Tokens"] = df_cleaned["Tokens_no_stopwords"].apply(
    lemmatize_tokens
)

In [189]:
df_cleaned["Review_lemmatized"] = df_cleaned["Lemmatized_Tokens"].apply(
    lambda tokens: " ".join(tokens)
)

In [190]:
df_cleaned[[
    "Review",
    "Tokens",
    "Tokens_no_stopwords",
    "Lemmatized_Tokens"
]].head()

,Review,Tokens,Tokens_no_stopwords,Lemmatized_Tokens
0,came here for the high tea. great service espe...,"[came, here, for, the, high, tea, ., great, se...","[came, high, tea, ., great, service, especiall...","[come, high, tea, ., great, service, especiall..."
1,"5 stars for the service, even though some of t...","[5, stars, for, the, service, ,, even, though,...","[5, stars, service, ,, even, though, staffs, n...","[5, star, service, ,, even, though, staff, nee..."
2,"hi, thank you for your service. but! i feel so...","[hi, ,, thank, you, for, your, service, ., but...","[hi, ,, thank, service, ., but, !, feel, so, s...","[hi, ,, thank, service, ., but, !, feel, so, s..."
3,i have the worse buffer dinner ever so far. th...,"[i, have, the, worse, buffer, dinner, ever, so...","[worse, buffer, dinner, ever, so, far, ., spre...","[bad, buffer, dinner, ever, so, far, ., spread..."
4,"that is are known 5 elmark "" 9h72 "" & kdk "" 3 ...","[that, is, are, known, 5, elmark, ``, 9h72, ``...","[known, 5, elmark, ``, 9h72, ``, &, kdk, ``, 3...","[know, 5, elmark, ``, 9h72, ``, &, kdk, ``, 3,..."


In [191]:
# keep the wanted columns only
df_cleaned = df_cleaned[[
    "Review",
    "Rating",
    "Lemmatized_Tokens",
    "Review_lemmatized",
    "Platform",
    "Restaurant"
]]

In [192]:
from pathlib import Path

# Find project root
project_dir = Path.cwd()

# Change this if your notebook is inside a subfolder
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Create cleaned data directory
cleaned_dir = project_dir / "data" / "cleaned_data"
cleaned_dir.mkdir(parents=True, exist_ok=True)

# Create raw data directory
raw_dir = project_dir / "data" / "raw_data"
raw_dir.mkdir(parents=True, exist_ok=True)

# Export raw file
output_file = raw_dir / "combined_reviews.csv"
df.to_csv(output_file, index=False)

# Export cleaned file
output_file = cleaned_dir / "cleaned_reviews.csv"
df_cleaned.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: c:\Users\user\Downloads\social_computing_assignment\data\cleaned_data\cleaned_reviews.csv
